# 02 - U-Net

Train a U-Net that gives every pixel one of three classes: 0 background, 1 nucleus interior, 2 nucleus boundary.
Kaggle: Accelerator GPU T4, Internet on. Output: `out/unet_best.pt`, `out/train_log.csv`, `out/loss_curve.png`, zipped as `results_02.zip`.

The only cell you write yourself is the `UNet` class. Everything else is ready.

In [ ]:
import glob, zipfile, urllib.request, io, time, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage import io as skio
from skimage.segmentation import find_boundaries
import torch
import torch.nn as nn
import torch.nn.functional as F

WORK = Path('/kaggle/working') if Path('/kaggle').exists() else Path('.')
DATA = Path('/tmp/nuclei_data')   # raw data is not saved as output
OUT = WORK / 'out'
DATA.mkdir(parents=True, exist_ok=True)
OUT.mkdir(exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0); np.random.seed(0)
print('device:', DEVICE)

URL = 'https://data.broadinstitute.org/bbbc/BBBC038/stage1_train.zip'
zip_path, train_dir = DATA / 'stage1_train.zip', DATA / 'stage1_train'
if not zip_path.exists():
    urllib.request.urlretrieve(URL, zip_path)
if not train_dir.exists():
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(train_dir)

# same split as notebook 01, read from the repository
SPLIT_URL = 'https://raw.githubusercontent.com/veliatees/nuclei-segmentation/main/results/split.csv'
split = pd.read_csv(io.BytesIO(urllib.request.urlopen(SPLIT_URL).read()))
paths = {Path(p).stem: p for p in glob.glob(str(train_dir / '**' / 'images' / '*.png'), recursive=True)}
split = split[split.id.isin(paths)].reset_index(drop=True)
print(split.groupby(['type', 'split']).size().unstack())

Load every image once into memory (670 small images fit easily). The target is the three-class map from notebook 01.
Images are scaled to 0..1 and then normalised per image, because brightness differs a lot between image types.

In [ ]:
def load(img_path):
    img = skio.imread(img_path)
    if img.ndim == 2:
        img = np.stack([img] * 3, axis=-1)
    img = img[..., :3]
    label = np.zeros(img.shape[:2], np.int32)
    for k, mp in enumerate(sorted(glob.glob(str(Path(img_path).parent.parent / 'masks' / '*.png'))), start=1):
        label[skio.imread(mp) > 0] = k
    return img, label

def three_class(label):
    t = np.zeros(label.shape, np.int64)
    t[label > 0] = 1
    t[find_boundaries(label, mode='inner') & (label > 0)] = 2
    return t

def normalise(img):
    x = img.astype(np.float32) / 255.0
    return (x - x.mean()) / (x.std() + 1e-6)

items = {}
for row in split.itertuples():
    img, label = load(paths[row.id])
    items[row.id] = (normalise(img).transpose(2, 0, 1), three_class(label), label)
train_ids = split[split.split == 'train'].id.tolist()
val_ids = split[split.split == 'val'].id.tolist()
print(len(train_ids), 'train,', len(val_ids), 'val')

Training uses random 256 x 256 crops with flips and 90-degree rotations, so the model sees many different pieces of each image.

In [ ]:
CROP = 256

class CropDataset(torch.utils.data.Dataset):
    def __init__(self, ids, crops_per_image=4):
        self.ids = ids
        self.n = crops_per_image
    def __len__(self):
        return len(self.ids) * self.n
    def __getitem__(self, i):
        x, t, _ = items[self.ids[i // self.n]]
        h, w = t.shape
        if h < CROP or w < CROP:   # pad small images
            ph, pw = max(0, CROP - h), max(0, CROP - w)
            x = np.pad(x, ((0, 0), (0, ph), (0, pw)), mode='reflect')
            t = np.pad(t, ((0, ph), (0, pw)), mode='constant')
            h, w = t.shape
        y0, x0 = np.random.randint(0, h - CROP + 1), np.random.randint(0, w - CROP + 1)
        x, t = x[:, y0:y0 + CROP, x0:x0 + CROP], t[y0:y0 + CROP, x0:x0 + CROP]
        k = np.random.randint(4)
        x, t = np.rot90(x, k, axes=(1, 2)), np.rot90(t, k)
        if np.random.rand() < 0.5:
            x, t = x[:, :, ::-1], t[:, ::-1]
        return torch.from_numpy(x.copy()), torch.from_numpy(t.copy())

train_loader = torch.utils.data.DataLoader(CropDataset(train_ids), batch_size=16, shuffle=True, num_workers=2, drop_last=True)
xb, tb = next(iter(train_loader))
print(xb.shape, tb.shape, torch.bincount(tb.flatten(), minlength=3))

## Your part: the U-Net

Build it from four small pieces:

1. `DoubleConv(in_ch, out_ch)`: Conv2d 3x3 (padding=1) -> BatchNorm2d -> ReLU, two times. Keeps height and width the same.
2. Encoder (down path): `DoubleConv`, then `MaxPool2d(2)` halves the size. Channels: 3 -> 32 -> 64 -> 128 -> 256, and a bottom block with 512.
3. Decoder (up path): `ConvTranspose2d(c, c // 2, kernel_size=2, stride=2)` doubles the size, then `torch.cat` with the encoder output of the same size (the skip connection), then `DoubleConv`.
4. Last layer: `Conv2d(32, n_classes, kernel_size=1)` gives 3 numbers (logits) per pixel.

The next cell checks your model: input (2, 3, 256, 256) must give output (2, 3, 256, 256).

In [ ]:
class UNet(nn.Module):
    def __init__(self, in_ch=3, n_classes=3, base=32):
        super().__init__()
        # TODO: write the layers here
        raise NotImplementedError('write the U-Net here')

    def forward(self, x):
        # TODO: encoder, bottom, decoder with skip connections, last 1x1 conv
        raise NotImplementedError

In [ ]:
model = UNet().to(DEVICE)
with torch.no_grad():
    out = model(torch.randn(2, 3, 256, 256, device=DEVICE))
assert out.shape == (2, 3, 256, 256), out.shape
print('ok, parameters:', sum(p.numel() for p in model.parameters()) / 1e6, 'million')

Loss: cross-entropy per pixel. Boundary pixels are rare but they are what separates touching nuclei, so they get weight 3.
Validation runs on full images (padded to a multiple of 16) and reports the loss and the foreground IoU (nucleus pixels vs background).

In [ ]:
EPOCHS = 40
loss_fn = nn.CrossEntropyLoss(weight=torch.tensor([1.0, 1.0, 3.0], device=DEVICE))
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

def pad16(x):
    h, w = x.shape[-2:]
    ph, pw = (-h) % 16, (-w) % 16
    return F.pad(x, (0, pw, 0, ph), mode='reflect'), h, w

@torch.no_grad()
def validate():
    model.eval()
    losses, inter, union = [], 0, 0
    for vid in val_ids:
        x, t, _ = items[vid]
        xt, h, w = pad16(torch.from_numpy(x)[None].to(DEVICE))
        logits = model(xt)[..., :h, :w]
        tt = torch.from_numpy(t)[None].to(DEVICE)
        losses.append(loss_fn(logits, tt).item())
        pred_fg, true_fg = logits.argmax(1) > 0, tt > 0
        inter += (pred_fg & true_fg).sum().item()
        union += (pred_fg | true_fg).sum().item()
    return float(np.mean(losses)), inter / max(union, 1)

log, best = [], 1e9
for epoch in range(1, EPOCHS + 1):
    model.train(); t0 = time.time(); run = []
    for xb, tb in train_loader:
        xb, tb = xb.to(DEVICE), tb.to(DEVICE)
        loss = loss_fn(model(xb), tb)
        opt.zero_grad(); loss.backward(); opt.step()
        run.append(loss.item())
    sched.step()
    val_loss, val_iou = validate()
    log.append(dict(epoch=epoch, train_loss=np.mean(run), val_loss=val_loss, val_fg_iou=val_iou, seconds=time.time() - t0))
    if val_loss < best:
        best = val_loss
        torch.save(model.state_dict(), OUT / 'unet_best.pt')
    print(f"epoch {epoch:2d}  train {np.mean(run):.3f}  val {val_loss:.3f}  fg IoU {val_iou:.3f}  {time.time() - t0:.0f}s")

In [ ]:
df = pd.DataFrame(log)
df.to_csv(OUT / 'train_log.csv', index=False)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(df.epoch, df.train_loss, label='train'); ax[0].plot(df.epoch, df.val_loss, label='val')
ax[0].set_xlabel('epoch'); ax[0].set_ylabel('loss'); ax[0].legend()
ax[1].plot(df.epoch, df.val_fg_iou); ax[1].set_xlabel('epoch'); ax[1].set_ylabel('val foreground IoU')
plt.tight_layout(); plt.savefig(OUT / 'loss_curve.png', dpi=110); plt.show()
print('best epoch:', int(df.loc[df.val_loss.idxmin(), 'epoch']))

In [ ]:
shutil.make_archive(str(WORK / 'results_02'), 'zip', OUT)
print(sorted(p.name for p in OUT.iterdir()))